In [1]:
import numpy as np
import pandas as pd
import torch

In [2]:
# Set random seed for reproducibility
np.random.seed(42)

In [13]:
# Generate synthetic dataset
n_days = 730  # Two years of daily data
dates = pd.date_range(start="2023-01-01", periods=n_days, freq="D")

In [14]:
# Function to create a trend (increasing or decreasing)
def generate_trend(n_days, start=0.8, end=1.2):
    return np.linspace(start, end, n_days)

In [16]:
# Function to create a seasonality with sinusoidal pattern
def generate_seasonality(n_days, amplitude=0.3):
    return 1 + amplitude * np.sin(2 * np.pi * np.arange(n_days) / 365)

In [21]:
# Function to get holiday dates from a DatetimeIndex object
def get_holiday_year_from_dates(dates, holidays_base, easter_dates):
    start_year = dates.year.min()  # Get the starting year from the DatetimeIndex
    end_year = dates.year.max()  # Get the ending year from the DatetimeIndex
    num_years = end_year - start_year + 1  # Calculate the number of years

    holiday_dates = {}
    for holiday, base_date in holidays_base.items():
        holiday_dates[holiday] = [f"{year}-{base_date}" for year in range(start_year, end_year + 1)]

    # Add the Easter dates to the holiday list, assuming it's a list of yyyy-mm-dd strings
    holiday_dates["Easter"] = easter_dates

    return holiday_dates

# Define holidays with just the base date (no year)
holidays_base = {
    "New Year's Day": "01-01",
    "Independence Day": "07-04",
    "Christmas": "12-25",
    "Black Friday": "11-24",
    "Cyber Monday": "11-27",
    "Singles' Day": "11-11",
    "Valentine's Day": "02-14",
    "Back to School": "08-15",
    "Brand Anniversary": "06-01"
}

# Provide your list of Easter dates here
easter_dates = ["2023-04-09", "2024-03-31"]

# Get the holiday dates from the DatetimeIndex
holidays = get_holiday_year_from_dates(dates, holidays_base, easter_dates)

# Function to generate holiday spikes
def generate_holiday_spikes(n_days, dates, holidays, ramp_up_days=5, min_spike=2.5, max_spike=5):
    spikes = np.ones(n_days)
    all_holidays = sum(holidays.values(), [])  # Flatten the holiday dates into one list

    for h_date in all_holidays:
        idx = (dates == h_date).argmax()
        for i in range(ramp_up_days):
            if idx - i >= 0:
                spikes[idx - i] += np.linspace(1.2, 2, ramp_up_days)[i]
        spikes[idx] += np.random.uniform(min_spike, max_spike)

    return spikes


In [22]:
# Function to randomly set zero-investment days
def apply_zero_investment(spend_array, probability=0.1):
    zero_days = np.random.rand(len(spend_array)) < probability
    spend_array[zero_days] = 0
    return spend_array

In [24]:
# Define different parameters for each media vehicle
tv_base = np.random.uniform(1000, 5000, n_days)
tv_trend = generate_trend(n_days, start=1.5, end=0.9)
tv_seasonality = generate_seasonality(n_days, amplitude=0.2)
tv_holiday_spikes = generate_holiday_spikes(n_days, dates, holidays, ramp_up_days = 10)
tv_spend = tv_base * tv_trend * tv_seasonality * tv_holiday_spikes

# Apply zero-investment conditions
tv_spend = apply_zero_investment(tv_spend, probability=0.15)

In [25]:
radio_base = np.random.uniform(500, 2000, n_days)
radio_trend = generate_trend(n_days, start=1.2, end=0.8)
radio_seasonality = generate_seasonality(n_days, amplitude=0.2)
radio_holiday_spikes = generate_holiday_spikes(n_days, dates, holidays, ramp_up_days = 10, min_spike=1.5, max_spike=3)
radio_spend = radio_base * radio_trend * radio_seasonality * radio_holiday_spikes

# Apply zero-investment conditions
radio_spend = apply_zero_investment(radio_spend, probability=0.20)

In [26]:
ooh_base = np.random.uniform(500, 1500, n_days)
ooh_trend = generate_trend(n_days, start=1.2, end=0.8)
ooh_seasonality = generate_seasonality(n_days, amplitude=0.1)
ooh_holiday_spikes = generate_holiday_spikes(n_days, dates, holidays, ramp_up_days = 10, min_spike=1.5, max_spike=2)
ooh_spend = ooh_base * ooh_trend * ooh_seasonality * ooh_holiday_spikes

# Apply zero-investment conditions
ooh_spend = apply_zero_investment(ooh_spend, probability=0.20)

In [27]:
meta_base = np.random.uniform(2000, 8000, n_days)
meta_trend = generate_trend(n_days, start=1.1, end=1.5)
meta_seasonality = generate_seasonality(n_days, amplitude=0.4)
meta_holiday_spikes = generate_holiday_spikes(n_days, dates, holidays, min_spike=3, max_spike=6)
meta_spend = meta_base * meta_trend * meta_seasonality * meta_holiday_spikes

In [28]:
google_base = np.random.uniform(5000, 10000, n_days)
google_trend = generate_trend(n_days, start=1.1, end=1.3)
google_seasonality = generate_seasonality(n_days, amplitude=0.4)
google_holiday_spikes = generate_holiday_spikes(n_days, dates, holidays, min_spike=3, max_spike=6)
google_spend = google_base * google_trend * google_seasonality * google_holiday_spikes

In [29]:
tiktok_base = np.random.uniform(3000, 5000, n_days)
tiktok_trend = generate_trend(n_days, start=1.1, end=1.8)
tiktok_seasonality = generate_seasonality(n_days, amplitude=0.4)
tiktok_holiday_spikes = generate_holiday_spikes(n_days, dates, holidays, min_spike=3, max_spike=6)
tiktok_spend = tiktok_base * tiktok_trend * tiktok_seasonality * tiktok_holiday_spikes

In [31]:
digital_others_base = np.random.uniform(1000, 2000, n_days)
digital_others_trend = generate_trend(n_days, start=0.6, end=1.5)
digital_others_seasonality = generate_seasonality(n_days, amplitude=0.4)
digital_others_holiday_spikes = generate_holiday_spikes(n_days, dates, holidays, min_spike=3, max_spike=6)
digital_others_spend = digital_others_base * digital_others_trend * digital_others_seasonality * digital_others_holiday_spikes

In [34]:
# Generate sales data

sales_baseline = np.random.uniform(2000, 3000, n_days)
sales_trend = generate_trend(n_days, start=0.9, end=1.7)
sales_seasonality = generate_seasonality(n_days, amplitude=0.1)
sales_holiday_spikes = generate_holiday_spikes(n_days, dates, holidays, ramp_up_days = 10, min_spike=1.5, max_spike=2)

# add noise
sales_noise = np.random.normal(0, 500, n_days)

# add media contribution
sales_from_media = (
    0.1 * tv_spend +
    0.05 * radio_spend +
    0.05 * ooh_spend +
    0.2 * meta_spend +
    0.3 * google_spend +
    0.2 * tiktok_spend +
    0.1 * digital_others_spend
)

# Sales = Adstock effect + seasonality + noise
sales_total = (sales_baseline + sales_from_media + sales_noise) * sales_trend * sales_seasonality * sales_holiday_spikes

In [12]:
# Create DataFrame
df = pd.DataFrame({
    "date": dates,
    "tv": tv_spend,
    "radio": radio_spend,
    "ooh": ooh_spend,
    "meta": meta_spend,
    "google": google_spend,
    "tiktok": tiktok_spend,
    "digital": digital_others_spend,
    "sales": sales_total
})

In [9]:
# Convert to Torch tensors and move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X = torch.tensor(df[["tv_spend", "radio_spend", "digital_spend", "seasonality"]].values, dtype=torch.float32).to(device)
y = torch.tensor(df["sales"].values, dtype=torch.float32).view(-1, 1).to(device)

print("Data loaded and moved to:", device)

Data loaded and moved to: cpu


In [10]:
# Visualize dataframe
df.head()

,date,tv_spend,radio_spend,digital_spend,seasonality,sales
0,2023-01-01,2498.160475,1079.153957,2985.594788,1008.606678,3207.916180
1,2023-01-02,4802.857226,1941.785846,6887.448321,1017.210806,5990.687105
2,2023-01-03,3927.975767,1858.025963,5991.183324,1025.809834,5849.087827
3,2023-01-04,3394.633937,793.686702,5138.392549,1034.401213,4469.289761
4,2023-01-05,1624.074562,604.041951,4152.982905,1042.982399,3479.520087
